# Semana 16 — RAG Evaluation: Faithfulness, Context e Capstone
**Fase 3: Advanced RAG, Search e Reranking** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/03-Phase-3-Advanced-RAG/notebooks/semana_16_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-16-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/03-Phase-3-Advanced-RAG/docs/Semana-16-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

In [ ]:
!pip -q install sentence-transformers rank_bm25

In [ ]:
import numpy as np, pandas as pd, re, torch, time
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)
MODELO_EMB = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_RERANK = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
MODELO_NLI = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"

## Passo 1 — Golden set
As queries com resposta de referência, mais perguntas **sem resposta na base**.

In [ ]:
# --- Corpus de demonstração: base de conhecimento fictícia de uma empresa ---
import random
def corpus_demo(seed=7):
    """Devolve (docs, consultas).
    docs: lista de {id, texto, area, ano}.
    consultas: lista de {q, resposta, relevantes: {doc_id: grau}}; grau 2 = contém a resposta, 1 = mesmo assunto."""
    rnd = random.Random(seed)
    fatos = {
        "ferias": [("O colaborador tem direito a 30 dias de férias após 12 meses de trabalho.", "Quantos dias de descanso anual eu tenho?"),
                   ("As férias podem ser divididas em até três períodos, um deles com no mínimo 14 dias.", "Posso fracionar meu período de descanso?"),
                   ("O pedido de férias deve ser feito no portal RH com 45 dias de antecedência.", "Com que antecedência solicito minha folga anual?")],
        "reembolso": [("Despesas de viagem são reembolsadas em até 10 dias úteis após o envio dos recibos.", "Em quanto tempo recebo de volta o que gastei viajando?"),
                      ("O limite diário de alimentação em viagens nacionais é de R$ 120.", "Qual o teto de gasto com refeição em deslocamento a trabalho?"),
                      ("Corridas de aplicativo só são reembolsadas entre 22h e 6h ou em deslocamentos a clientes.", "A empresa paga meu Uber?")],
        "seguranca": [("Senhas corporativas expiram a cada 90 dias e exigem autenticação em dois fatores.", "De quanto em quanto tempo preciso trocar minha credencial?"),
                      ("Dados de clientes classificados como confidenciais não podem ser enviados a ferramentas externas de IA.", "Posso colar informações de cliente em um chatbot público?"),
                      ("Incidentes de segurança devem ser reportados ao SOC em até 2 horas.", "Qual o prazo para avisar sobre um vazamento?")],
        "ti": [("Notebooks são substituídos a cada 36 meses mediante chamado no service desk.", "Quando posso trocar meu computador?"),
               ("O acesso à VPN exige o cliente GlobalConnect e certificado pessoal.", "Como me conecto à rede da empresa de casa?"),
               ("Licenças de software adicionais precisam de aprovação do gestor e do time de compras.", "Como consigo um programa novo?")],
        "beneficios": [("O plano de saúde cobre dependentes legais sem custo adicional até 21 anos.", "Meu filho pode entrar no convênio médico?"),
                       ("O vale-refeição é creditado no último dia útil de cada mês.", "Em que dia cai o cartão de alimentação?"),
                       ("A empresa subsidia 50% de cursos de idiomas, limitado a R$ 400 por mês.", "Existe ajuda para estudar inglês?")],
    }
    codigos = [("O código de erro VPN-4031 indica certificado pessoal expirado; renove pelo portal de identidades.", "ti", "O que significa o erro VPN-4031?"),
               ("O formulário RH-207 é usado para solicitar licença não remunerada.", "ferias", "Para que serve o formulário RH-207?"),
               ("O centro de custo CC-8812 deve ser usado para despesas de treinamento externo.", "reembolso", "Qual centro de custo uso: CC-8812 é para quê?"),
               ("A política SEG-014 define a classificação de dados em público, interno e confidencial.", "seguranca", "O que diz a política SEG-014?")]
    projetos = ["Atlas", "Boreal", "Cometa", "Delta", "Éden"]
    armadilhas = ["O time do projeto {p} volta de férias em janeiro e retoma as entregas do trimestre.",
                  "No projeto {p}, o reembolso ao cliente foi aprovado após a revisão do contrato.",
                  "A equipe do projeto {p} trocou a senha do ambiente de testes durante a migração.",
                  "O projeto {p} comprou notebooks novos para o laboratório de dados.",
                  "O plano de saúde financeira do projeto {p} prevê corte de 10% nos custos de nuvem."]
    ruido = ["Este procedimento foi revisado pelo comitê interno.", "Dúvidas devem ser encaminhadas ao gestor imediato.",
             "A política vale para todas as unidades do grupo.", "Exceções exigem aprovação formal da diretoria."]
    docs, consultas = [], []
    def novo(texto, area):
        docs.append({"id": len(docs), "texto": texto, "area": area, "ano": rnd.choice([2023, 2024, 2025])}); return docs[-1]["id"]
    por_area = {}
    for area, itens in fatos.items():
        for fato, pergunta in itens:
            i = novo(f"{fato} {rnd.choice(ruido)}", area); por_area.setdefault(area, []).append(i)
            consultas.append({"q": pergunta, "resposta": fato, "relevantes": {i: 2}, "area": area})
    for texto, area, pergunta in codigos:
        i = novo(texto, area); por_area[area].append(i)
        consultas.append({"q": pergunta, "resposta": texto, "relevantes": {i: 2}, "area": area})
    for c in consultas:
        for j in por_area[c["area"]]:
            c["relevantes"].setdefault(j, 1)
    for p in projetos:
        for a in armadilhas:
            novo(a.format(p=p), "projetos")
    return docs, consultas

def manual_demo(docs):
    """Junta os documentos de política em um manual longo em Markdown (para exercícios de chunking)."""
    titulos = {"ferias": "Férias e licenças", "reembolso": "Viagens e reembolsos", "seguranca": "Segurança da informação",
               "ti": "Equipamentos e acesso", "beneficios": "Benefícios"}
    partes = ["# Manual do Colaborador\n\nEste manual reúne as políticas internas vigentes."]
    for area, titulo in titulos.items():
        frases = [d["texto"] for d in docs if d["area"] == area]
        partes.append(f"## {titulo}\n\n" + "\n\n".join(" ".join(frases[i:i + 2]) for i in range(0, len(frases), 2)))
    return "\n\n".join(partes)

docs, consultas = corpus_demo()
textos = [d["texto"] for d in docs]
sem_resposta = ["Qual é a política de estacionamento da sede?", "Quantos dias de licença-paternidade a empresa oferece?",
                "Qual o valor do bônus anual?", "A empresa permite trabalho remoto de outro país?", "Qual o horário de funcionamento do refeitório?"]
golden = [dict(c, tem_resposta=True) for c in consultas] + [{"q": q, "resposta": None, "relevantes": {}, "tem_resposta": False} for q in sem_resposta]
print(len(golden), "casos |", sum(g["tem_resposta"] for g in golden), "com resposta |", len(sem_resposta), "sem resposta")

## Passo 2 — O sistema sob teste (pipeline das Semanas 13 e 14)

In [ ]:
emb_model = SentenceTransformer(MODELO_EMB); reranker = CrossEncoder(MODELO_RERANK)
embed = lambda ts: emb_model.encode(list(ts), normalize_embeddings=True, convert_to_numpy=True)
E = embed(textos); tk = lambda t: re.findall(r"\w+", t.lower()); bm25 = BM25Okapi([tk(t) for t in textos])

def recuperar(q, k=3, rerank=True, n_cand=15):
    pts = {}
    for lista in (np.argsort(-bm25.get_scores(tk(q)))[:20], np.argsort(-(E @ embed([q])[0]))[:20]):
        for pos, d in enumerate(lista, start=1): pts[int(d)] = pts.get(int(d), 0) + 1 / (60 + pos)
    cand = sorted(pts, key=pts.get, reverse=True)[:n_cand]
    if rerank:
        s = np.asarray(reranker.predict([(q, textos[i]) for i in cand])).reshape(-1); cand = [cand[i] for i in np.argsort(-s)]
    return cand[:k]

tok = AutoTokenizer.from_pretrained(MODELO_LLM)
llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
RECUSA = "Não encontrei essa informação."
SISTEMA = ("Você é um assistente interno. Responda SOMENTE com base no CONTEXTO, em uma frase. "
           f"Se a resposta não estiver no contexto, responda exatamente: '{RECUSA}'")

def responder(q, ids):
    contexto = "\n".join(f"[{n + 1}] {textos[i]}" for n, i in enumerate(ids))
    msgs = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPERGUNTA: {q}"}]
    ent = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt", add_special_tokens=False).to(device)
    with torch.no_grad():
        out = llm.generate(**ent, max_new_tokens=60, do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, ent["input_ids"].shape[1]:], skip_special_tokens=True).strip()

ids = recuperar(golden[0]["q"]); print(golden[0]["q"], "->", responder(golden[0]["q"], ids))

## Passo 3 — Retrieval metrics
- `recall_contexto`: 1 se o documento de grau 2 está entre os recuperados.
- `precisao_contexto`: fração dos recuperados com grau ≥ 1.

In [ ]:
def recall_contexto(ids, rel):
    return float(any(rel.get(i, 0) == 2 for i in ids))

def precisao_contexto(ids, rel):
    return sum(rel.get(i, 0) >= 1 for i in ids) / len(ids)

## Passo 4 — Faithfulness com NLI
O modelo de NLI recebe (premissa = contexto, hipótese = resposta) e devolve probabilidades de *entailment*, *neutral* e *contradiction*. Faithfulness = probabilidade de *entailment*.

In [ ]:
tok_nli = AutoTokenizer.from_pretrained(MODELO_NLI)
nli = AutoModelForSequenceClassification.from_pretrained(MODELO_NLI).to(device).eval()
IDX_ENT = next(i for i, l in nli.config.id2label.items() if str(l).lower().startswith("entail"))

@torch.no_grad()
def fidelidade(resposta, contexto):
    ent = tok_nli(contexto, resposta, truncation=True, max_length=512, return_tensors="pt").to(device)
    probs = nli(**ent).logits.softmax(-1)[0]
    return float(probs[int(IDX_ENT)])

ctx = "O limite diário de alimentação em viagens nacionais é de R$ 120."
print("fiel      :", round(fidelidade("O teto de refeição em viagens dentro do país é R$ 120 por dia.", ctx), 3))
print("alucinado :", round(fidelidade("O teto de refeição em viagens dentro do país é R$ 300 por dia.", ctx), 3))

## Passo 5 — Correção da resposta e recusa
- `correcao`: cosine similarity entre a resposta e a resposta de referência.
- `recusou`: a resposta contém a frase de recusa?

In [ ]:
def correcao(resposta, referencia):
    a, b = embed([resposta, referencia])
    return float(a @ b)

def recusou(resposta):
    return "não encontrei" in resposta.lower()

## Passo 6 — Rodar a avaliação
Para cada caso do golden set: recuperar, responder e calcular as métricas aplicáveis.

In [ ]:
def avaliar(k=3, rerank=True):
    linhas = []
    for g in golden:
        ids = recuperar(g["q"], k, rerank); resp = responder(g["q"], ids); ctx = " ".join(textos[i] for i in ids)
        linha = {"q": g["q"], "resposta": resp, "tem_resposta": g["tem_resposta"], "recusou": recusou(resp)}
        if g["tem_resposta"]:
            linha.update(recall_ctx=recall_contexto(ids, g["relevantes"]), precisao_ctx=precisao_contexto(ids, g["relevantes"]),
                         fidelidade=fidelidade(resp, ctx), correcao=correcao(resp, g["resposta"]))
        linhas.append(linha)
    return pd.DataFrame(linhas)

t0 = time.time(); base = avaliar(k=3, rerank=True); print(f"{time.time() - t0:.0f}s"); base.head()

## Passo 7 — Resumo: uma linha por configuração
Agregue: médias das métricas nos casos com resposta, **recusa indevida** (recusou tendo resposta) e **recusa correta** (recusou quando não havia resposta).

In [ ]:
def resumir(df):
    com, sem = df[df["tem_resposta"]], df[~df["tem_resposta"]]
    return {"recall_ctx": com["recall_ctx"].mean(), "precisao_ctx": com["precisao_ctx"].mean(),
            "fidelidade": com["fidelidade"].mean(), "correcao": com["correcao"].mean(),
            "recusa_indevida": com["recusou"].mean(), "recusa_correta": sem["recusou"].mean()}

configs = {"k=3 com rerank": base, "k=3 sem rerank": avaliar(k=3, rerank=False), "k=1 com rerank": avaliar(k=1, rerank=True)}
pd.DataFrame({nome: resumir(df) for nome, df in configs.items()}).T.round(3)

Qual configuração você colocaria em produção? Que métrica pesou mais na decisão?

## Passo 8 — Matriz de diagnóstico
Classifique cada caso com resposta em uma das quatro células: contexto certo ou ausente × resposta fiel ou infiel (threshold de faithfulness 0,5).

In [ ]:
def diagnosticar(linha, limiar=0.5):
    ctx_ok = linha["recall_ctx"] == 1; fiel = linha["fidelidade"] >= limiar
    if ctx_ok and fiel: return "sucesso"
    if ctx_ok and not fiel: return "falha de geração"
    if not ctx_ok and linha["recusou"]: return "recusa por falta de contexto"
    return "falha de recuperação"

com = base[base["tem_resposta"]].copy(); com["diagnostico"] = com.apply(diagnosticar, axis=1)
print(com["diagnostico"].value_counts(), "\n")
for _, l in com[com["diagnostico"] != "sucesso"].head(3).iterrows():
    print(f"[{l['diagnostico']}] {l['q']}\n   -> {l['resposta']}\n")

## Passo 9 (opcional) — O mesmo com RAGAS
RAGAS calcula métricas equivalentes usando um LLM como juiz, o que exige uma chave de API ou um LLM local configurado. Consulte a documentação atual do pacote antes de usar: a API muda entre versões.

## Fechamento da Fase 3
Escreva o relatório de uma página pedido no entregável. Ele é o modelo do documento que você apresentaria a um comitê de arquitetura para aprovar um RAG em produção.